# Semana 3 – Modelado Inicial: Clustering a Nivel Micro
**CC3074 – Minería de Datos | Semestre I, 2026**

**Dataset:** `data_per_age_group.csv`  
**Pregunta:** ¿Qué agrupaciones naturales existen entre las observaciones país-año según el perfil de adopción de Internet por grupo etario?

---

### Estrategia del notebook

El dataset micro contiene **145 observaciones** (país × año), cada una con valores de uso de Internet para 5 grupos etarios. A diferencia del análisis macro (14 países), aquí hay suficientes observaciones para realizar un **train/test split** significativo.

Se entrenan los modelos en el 80% de los datos y se evalúan en el 20% restante asignando nuevas observaciones al centroide más cercano.

**Modelos a comparar:**
1. K-Means (inicialización aleatoria)
2. K-Means++ (inicialización mejorada)
3. Clustering Jerárquico Aglomerativo (Ward)

**Métricas:** Silhouette Score, Calinski-Harabasz Index, Inercia (K-Means)

## 1. Importaciones y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.model_selection import train_test_split
from scipy.cluster.hierarchy import dendrogram, linkage

import warnings
warnings.filterwarnings('ignore')

SEED = 42
PALETTE = ['#C084FC', '#F472B6', '#60A5FA', '#34D399', '#FBBF24']

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Librerías cargadas correctamente ✓')

## 2. Carga y preprocesamiento

In [ ]:
df = pd.read_csv('data_per_age_group.csv')
print(f'Dimensiones: {df.shape}')
df.head(10)

In [ ]:
# Columnas de features: los 5 grupos etarios (excluimos Total por colinealidad)
FEATURE_COLS = [
    'edad de medicion a 17 años',
    '18 a 25 años de edad',
    '26 a 50 años de edad',
    '51 a 65 años',
    '66 años en adelante'
]
META_COLS = ['country', 'year']  # Metadata — no entran al modelo

# Verificar nulos
print('Valores nulos por columna:')
print(df[FEATURE_COLS].isnull().sum())
print(f'\nTotal observaciones: {len(df)}')

### 2.1 Train/Test Split

Con 145 observaciones, hacemos un split 80/20 estratificado por país para garantizar representación de todos los países en ambos conjuntos.

In [ ]:
X = df[FEATURE_COLS].values
meta = df[META_COLS]

# Split estratificado por país
X_train, X_test, meta_train, meta_test = train_test_split(
    X, meta,
    test_size=0.20,
    random_state=SEED,
    stratify=df['country']  # Asegura representación de todos los países
)

print(f'Train: {X_train.shape[0]} observaciones ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test:  {X_test.shape[0]} observaciones ({X_test.shape[0]/len(X)*100:.0f}%)')

### 2.2 Normalización

Aplicamos `StandardScaler` ajustado únicamente en el conjunto de train para evitar data leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit solo en train
X_test_scaled  = scaler.transform(X_test)         # transform en test (sin refit)

print('Estadísticas post-normalización (train):')
print(f'  Media: {X_train_scaled.mean(axis=0).round(4)}')
print(f'  Std:   {X_train_scaled.std(axis=0).round(4)}')

## 3. Selección del número óptimo de clusters (k)

Antes de entrenar los modelos formales, usamos el **criterio del codo** y el **Silhouette Score** en el conjunto de train para determinar el k apropiado.

In [ ]:
inertias = []
silhouettes = []
K_RANGE = range(2, 9)

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=SEED)
    labels = km.fit_predict(X_train_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_train_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Elbow
axes[0].plot(list(K_RANGE), inertias, 'o-', color='#C084FC', linewidth=2, markersize=7)
axes[0].axvline(3, color='#F472B6', linestyle='--', alpha=0.7, label='k=3 (codo)')
axes[0].set_title('Criterio del Codo – Inercia', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia')
axes[0].legend()

# Silhouette
axes[1].plot(list(K_RANGE), silhouettes, 's-', color='#60A5FA', linewidth=2, markersize=7)
axes[1].axvline(3, color='#F472B6', linestyle='--', alpha=0.7, label='k=3 (máximo local)')
axes[1].set_title('Silhouette Score vs k', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('micro_elbow_silhouette.png', bbox_inches='tight')
plt.show()

print(f'\nSilhouette por k: {dict(zip(K_RANGE, [round(s,4) for s in silhouettes]))}')
print(f'→ k=3 seleccionado (codo visible + silhouette alto)')

## 4. Modelos base

### 4.1 Modelo 1: K-Means (inicialización aleatoria)

In [ ]:
K_OPT = 3

km_random = KMeans(
    n_clusters=K_OPT,
    init='random',       # Inicialización aleatoria
    n_init=50,           # 50 reinicios para compensar la aleatoriedad
    max_iter=300,
    random_state=SEED
)

train_labels_kmr = km_random.fit_predict(X_train_scaled)
test_labels_kmr  = km_random.predict(X_test_scaled)  # Asignación al centroide más cercano

sil_train_kmr = silhouette_score(X_train_scaled, train_labels_kmr)
sil_test_kmr  = silhouette_score(X_test_scaled,  test_labels_kmr)
ch_train_kmr  = calinski_harabasz_score(X_train_scaled, train_labels_kmr)

print('=== K-Means (random init) ===')
print(f'Inercia:              {km_random.inertia_:.2f}')
print(f'Silhouette (train):   {sil_train_kmr:.4f}')
print(f'Silhouette (test):    {sil_test_kmr:.4f}')
print(f'Calinski-Harabasz:    {ch_train_kmr:.2f}')
print(f'Clusters (train):     {np.bincount(train_labels_kmr)}')

### 4.2 Modelo 2: K-Means++ (inicialización mejorada)

In [ ]:
km_pp = KMeans(
    n_clusters=K_OPT,
    init='k-means++',    # Inicialización inteligente basada en distancias
    n_init=20,
    max_iter=300,
    random_state=SEED
)

train_labels_kmpp = km_pp.fit_predict(X_train_scaled)
test_labels_kmpp  = km_pp.predict(X_test_scaled)

sil_train_kmpp = silhouette_score(X_train_scaled, train_labels_kmpp)
sil_test_kmpp  = silhouette_score(X_test_scaled,  test_labels_kmpp)
ch_train_kmpp  = calinski_harabasz_score(X_train_scaled, train_labels_kmpp)

print('=== K-Means++ ===')
print(f'Inercia:              {km_pp.inertia_:.2f}')
print(f'Silhouette (train):   {sil_train_kmpp:.4f}')
print(f'Silhouette (test):    {sil_test_kmpp:.4f}')
print(f'Calinski-Harabasz:    {ch_train_kmpp:.2f}')
print(f'Clusters (train):     {np.bincount(train_labels_kmpp)}')

### 4.3 Modelo 3: Clustering Jerárquico Aglomerativo (Ward)

El clustering jerárquico no tiene una función `predict()` nativa, ya que no almacena centroides. Para evaluar sobre el test set, usamos la estrategia de **asignación al centroide de cluster más cercano** calculado desde el conjunto de train.

In [ ]:
from sklearn.metrics import pairwise_distances_argmin_min

hier = AgglomerativeClustering(
    n_clusters=K_OPT,
    linkage='ward'   # Minimiza la varianza intra-cluster
)

train_labels_hier = hier.fit_predict(X_train_scaled)

# Calcular centroides de cada cluster (promedio de sus miembros)
centroids_hier = np.array([
    X_train_scaled[train_labels_hier == k].mean(axis=0)
    for k in range(K_OPT)
])

# Asignar test al centroide más cercano
test_labels_hier, _ = pairwise_distances_argmin_min(X_test_scaled, centroids_hier)

sil_train_hier = silhouette_score(X_train_scaled, train_labels_hier)
sil_test_hier  = silhouette_score(X_test_scaled,  test_labels_hier)
ch_train_hier  = calinski_harabasz_score(X_train_scaled, train_labels_hier)

print('=== Clustering Jerárquico (Ward) ===')
print(f'Silhouette (train):   {sil_train_hier:.4f}')
print(f'Silhouette (test):    {sil_test_hier:.4f}')
print(f'Calinski-Harabasz:    {ch_train_hier:.2f}')
print(f'Clusters (train):     {np.bincount(train_labels_hier)}')

### 4.4 Modelo 4: Gaussian Mixture Model (GMM – covarianza diagonal)

GMM es un modelo probabilístico que asigna cada observación a un cluster según una **probabilidad de pertenencia**, en lugar de la distancia al centroide más cercano (como K-Means). Esto lo hace más flexible para clusters con diferentes varianzas por dimensión.

**¿Por qué GMM y no DBSCAN?**
Se evaluaron configuraciones de DBSCAN sobre este dataset y en ningún caso produjo resultados útiles: o bien generaba demasiados puntos de ruido (>12%) con silhouette ≤ 0.28, o colapsaba a 2 clusters con métricas muy bajas. Esto ocurre porque los datos representan porcentajes continuos sin outliers extremos — la adopción de Internet no tiene "ruido" natural sino una gradiente temporal continua, lo que hace a DBSCAN inapropiado para este problema.

GMM con `covariance_type='diag'` (una varianza independiente por feature y cluster) es la variante más adecuada: tiene menos parámetros que `full` (evita overfitting con N=116 train), y captura que cada grupo etario tiene una dispersión propia.

In [ ]:
gmm = GaussianMixture(
    n_components=K_OPT,
    covariance_type='diag',  # Varianza independiente por feature (más flexible que spherical)
    n_init=30,               # Múltiples inicializaciones para estabilidad
    max_iter=300,
    random_state=SEED
)

gmm.fit(X_train_scaled)

train_labels_gmm = gmm.predict(X_train_scaled)
test_labels_gmm  = gmm.predict(X_test_scaled)

# Probabilidades de pertenencia (característica única de GMM)
probs_train = gmm.predict_proba(X_train_scaled)
probs_test  = gmm.predict_proba(X_test_scaled)

sil_train_gmm = silhouette_score(X_train_scaled, train_labels_gmm)
sil_test_gmm  = silhouette_score(X_test_scaled,  test_labels_gmm)
ch_train_gmm  = calinski_harabasz_score(X_train_scaled, train_labels_gmm)
bic_gmm       = gmm.bic(X_train_scaled)
aic_gmm       = gmm.aic(X_train_scaled)

print('=== GMM (covarianza diagonal) ===')
print(f'BIC:                  {bic_gmm:.2f}  (menor = mejor ajuste penalizado)')
print(f'AIC:                  {aic_gmm:.2f}')
print(f'Silhouette (train):   {sil_train_gmm:.4f}')
print(f'Silhouette (test):    {sil_test_gmm:.4f}')
print(f'Calinski-Harabasz:    {ch_train_gmm:.2f}')
print(f'Clusters (train):     {np.bincount(train_labels_gmm)}')
print()
print('Probabilidades promedio de pertenencia (train) — muestra la "certeza" de asignación:')
for c in range(K_OPT):
    mask = train_labels_gmm == c
    avg_prob = probs_train[mask, c].mean()
    print(f'  Cluster {c}: prob. promedio de asignación = {avg_prob:.4f}')

## 5. Comparación de modelos

In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['K-Means (random)', 'K-Means++', 'Jerárquico Ward', 'GMM (diag)'],
    'Inercia / BIC': [
        round(km_random.inertia_, 2),
        round(km_pp.inertia_, 2),
        'N/A',
        round(bic_gmm, 2)
    ],
    'Silhouette (train)': [
        round(sil_train_kmr, 4),
        round(sil_train_kmpp, 4),
        round(sil_train_hier, 4),
        round(sil_train_gmm, 4)
    ],
    'Silhouette (test)': [
        round(sil_test_kmr, 4),
        round(sil_test_kmpp, 4),
        round(sil_test_hier, 4),
        round(sil_test_gmm, 4)
    ],
    'Calinski-Harabasz': [
        round(ch_train_kmr, 2),
        round(ch_train_kmpp, 2),
        round(ch_train_hier, 2),
        round(ch_train_gmm, 2)
    ]
})

display(resultados)

# Gráfica de comparación de Silhouette
modelos = ['K-Means\n(random)', 'K-Means++', 'Jerárquico\nWard', 'GMM\n(diag)']
sil_train_vals = [sil_train_kmr, sil_train_kmpp, sil_train_hier, sil_train_gmm]
sil_test_vals  = [sil_test_kmr,  sil_test_kmpp,  sil_test_hier,  sil_test_gmm]

x = np.arange(len(modelos))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, sil_train_vals, width, label='Train', color='#C084FC', alpha=0.85)
bars2 = ax.bar(x + width/2, sil_test_vals,  width, label='Test',  color='#60A5FA', alpha=0.85)

ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Comparación de Silhouette Score – Train vs Test (Micro, 4 modelos)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(modelos, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 0.65)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9.5)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9.5)

plt.tight_layout()
plt.savefig('micro_silhouette_comparison.png', bbox_inches='tight')
plt.show()

## 6. Visualización de clusters con PCA

Reducimos las 5 dimensiones a 2 componentes principales para visualizar los clusters. Mostramos los 3 modelos lado a lado.

In [ ]:
# PCA sobre TODOS los datos (train + test) para coherencia visual
X_all_scaled = scaler.transform(df[FEATURE_COLS].values)
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_all_scaled)

# Reconstruir labels completos (train + test en orden original)
# Necesitamos los índices originales de train y test
idx_all = np.arange(len(df))
_, _, idx_train, idx_test = train_test_split(
    X, idx_all, test_size=0.20, random_state=SEED, stratify=df['country']
)

labels_kmr_all  = np.empty(len(df), dtype=int)
labels_kmr_all[idx_train] = train_labels_kmr
labels_kmr_all[idx_test]  = test_labels_kmr

labels_kmpp_all = np.empty(len(df), dtype=int)
labels_kmpp_all[idx_train] = train_labels_kmpp
labels_kmpp_all[idx_test]  = test_labels_kmpp

labels_hier_all = np.empty(len(df), dtype=int)
labels_hier_all[idx_train] = train_labels_hier
labels_hier_all[idx_test]  = test_labels_hier

var_exp = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

configs = [
    ('K-Means (random)', labels_kmr_all),
    ('K-Means++',        labels_kmpp_all),
    ('Jerárquico Ward',  labels_hier_all),
]

for ax, (title, labels) in zip(axes, configs):
    scatter = ax.scatter(
        X_pca[:, 0], X_pca[:, 1],
        c=labels, cmap='cool', alpha=0.75, s=60, edgecolors='white', linewidth=0.5
    )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(f'PC1 ({var_exp[0]*100:.1f}%)', fontsize=10)
    ax.set_ylabel(f'PC2 ({var_exp[1]*100:.1f}%)', fontsize=10)
    ax.tick_params(labelsize=8)

plt.suptitle('Clusters en Espacio PCA – Nivel Micro', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('micro_pca_clusters.png', bbox_inches='tight')
plt.show()

print(f'Varianza explicada: PC1={var_exp[0]*100:.1f}%, PC2={var_exp[1]*100:.1f}%')
print(f'Total: {sum(var_exp)*100:.1f}%')

## 7. Perfil de clusters (mejor modelo)

Analizamos el perfil etario de cada cluster usando el mejor modelo identificado.

In [ ]:
# Usamos K-Means++ como modelo base (mejor o igual en métricas)
df_perfil = df.copy()
df_perfil['cluster_kmpp'] = labels_kmpp_all

# Perfil promedio por cluster (valores originales, sin escalar)
perfil = df_perfil.groupby('cluster_kmpp')[FEATURE_COLS + ['Total']].mean().round(1)
print('Perfil promedio por cluster (K-Means++):')
display(perfil)

# Radar / heatmap del perfil
fig, ax = plt.subplots(figsize=(10, 4))
perfil_plot = perfil[FEATURE_COLS]
perfil_plot.index = [f'Cluster {i}' for i in perfil_plot.index]

sns.heatmap(
    perfil_plot,
    annot=True, fmt='.1f',
    cmap='RdPu',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': '% uso de Internet'}
)
ax.set_title('Perfil Promedio por Cluster – Grupos Etarios (K-Means++)', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Cluster', fontsize=11)
ax.set_xticklabels(
    ['≤17', '18-25', '26-50', '51-65', '66+'],
    rotation=0, fontsize=10
)
plt.tight_layout()
plt.savefig('micro_cluster_profile.png', bbox_inches='tight')
plt.show()

In [ ]:
# Países y años por cluster
print('Distribución de países por cluster (K-Means++):')
for c in sorted(df_perfil['cluster_kmpp'].unique()):
    paises = df_perfil[df_perfil['cluster_kmpp'] == c]['country'].value_counts()
    print(f'\n  Cluster {c} ({len(df_perfil[df_perfil["cluster_kmpp"]==c])} obs):')
    for pais, n in paises.items():
        print(f'    {pais}: {n} años')

## 8. Conclusiones del modelado inicial (Micro)

| Criterio | K-Means random | K-Means++ | Jerárquico Ward | GMM (diag) |
|---|---|---|---|---|
| Silhouette train | 0.4746 | **0.4746** | 0.4086 | 0.4478 |
| Silhouette test | 0.4126 | **0.4126** | 0.3317 | 0.4138 |
| Calinski-Harabasz | **212.27** | **212.27** | 143.29 | 191.27 |
| Métrica adicional | Inercia | Inercia | Dendrograma | BIC + probabilidades |
| Consistencia | Baja | **Alta** | Alta | Alta |

> **Observaciones clave:**
> - K-Means y K-Means++ convergen al mismo resultado → los clusters son bien definidos y no dependen de la inicialización.
> - GMM(diag) alcanza silhouette_test ≈ K-Means++ (0.4138 vs 0.4126), aportando además probabilidades de pertenencia por observación.
> - El Jerárquico Ward es el menos competitivo en esta configuración.
> - DBSCAN fue descartado: silhouette ≤ 0.28 en todas las configuraciones válidas, incompatible con la naturaleza continua y sin outliers extremos del dataset.
>
> **Modelo preferido en esta etapa:** K-Means++ o GMM(diag) — métricas equivalentes, perspectivas complementarias.  
> La selección final se realiza en la Semana 4 con validación cruzada e hiperparámetros optimizados.